# Export to Hugging Face Hub

Uploads your fine-tuned model (`merged/` folder) to Hugging Face, so anyone can load it with **one line**:
`AutoModelForCausalLM.from_pretrained("your-name/your-model")`.

**Steps:** 1. config → 2. check the files → 3. set greedy decoding defaults → 4. test it once → 5. write a model card → 6. log in → 7. upload → 8. load it back.

You need a free Hugging Face account and an access token with **write** permission (huggingface.co → Settings → Access Tokens).


## 1. Config
Set `HF_REPO_ID` to `your-hf-username/model-name`. Keep `PRIVATE = True` until you've checked the upload; you can make it public later on the website.

In [ ]:
import json, sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(REPO_ROOT))
NOTEBOOKS = REPO_ROOT / "notebooks"

RUN_NAME = "sft_qwen2.5-0.5b"            # change to "sft_qwen2.5-0.5b_opt" after the optimized run
MERGED_DIR = NOTEBOOKS / "outputs" / RUN_NAME / "merged"
RESULTS_DIR = NOTEBOOKS / "data" / "eval" / "results" / RUN_NAME

HF_REPO_ID = "SwapniL-pr1117/qwen2.5-0.5b-instruct-fc"   # <-- change this
PRIVATE = True

print("Model folder:", MERGED_DIR, "| exists:", MERGED_DIR.exists())

## 2. Check what we're uploading
The folder must contain the weights (`model.safetensors`), the architecture (`config.json`) and the tokenizer. The safetensors header tells us the number type and the parameter count without loading the model.

In [ ]:
import math
import struct

for f in sorted(MERGED_DIR.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size / 1024**2:8.1f} MB")

with open(MERGED_DIR / "model.safetensors", "rb") as fh:
    header = json.loads(fh.read(struct.unpack("<Q", fh.read(8))[0]))
header.pop("__metadata__", None)
n_params = sum(math.prod(t["shape"]) for t in header.values())
dtypes = {t["dtype"] for t in header.values()}
print(f"\n{len(header)} tensors | {n_params / 1e6:.0f}M parameters | dtype: {dtypes}")

## 3. Set greedy decoding as the default
`merged/generation_config.json` still holds Qwen's **chat** defaults (`temperature 0.7`, `top_p 0.8`, `repetition_penalty 1.1`, sampling on).
Anyone calling plain `model.generate()` would get those, i.e. random and lower accuracy than our greedy eval.
`repetition_penalty` is especially bad for JSON, which repeats `"`, `{` and `:` all the time.
We overwrite it with greedy decoding so the model behaves like it did in evaluation **out of the box**.


In [ ]:
gen_path = MERGED_DIR / "generation_config.json"
gen = json.loads(gen_path.read_text())
print("before:", {k: gen.get(k) for k in ("do_sample", "temperature", "top_p", "top_k", "repetition_penalty")})

for k in ("temperature", "top_p", "top_k"):      # only meaningful when sampling
    gen.pop(k, None)
gen["do_sample"] = False
gen["repetition_penalty"] = 1.0

gen_path.write_text(json.dumps(gen, indent=2))
print("after: ", {k: gen.get(k) for k in ("do_sample", "temperature", "top_p", "top_k", "repetition_penalty")})


## 4. Test it once before uploading
Load with plain `transformers` and ask one question in the **trained prompt format** (`build_messages`: system prompt + tools + request). We expect a JSON tool call back.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.prompts import build_messages
from src.evaluate import extract_json

tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
model = AutoModelForCausalLM.from_pretrained(MERGED_DIR, dtype="auto", device_map="auto")

record = {
    "query": "What's the weather like in Pune right now?",
    "tools": [{"name": "get_weather", "description": "Current weather for a city.",
               "parameters": {"city": {"type": "str", "description": "City name"}}}],
}
inputs = tokenizer.apply_chat_template(build_messages(record), add_generation_prompt=True,
                                       return_tensors="pt", return_dict=True).to(model.device)
out = getattr(model, "generate")(
    **inputs,
    max_new_tokens=128,
    do_sample=False,
)
answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("MODEL :", answer)
print("PARSED:", extract_json(answer))

del model
torch.cuda.empty_cache()

## 5. Write the model card (`README.md`)
The model card is the page people see on Hugging Face. The block at the top (between `---`) is metadata: base model, license and tags make it searchable.
The **prompt format** section matters most: without it, nobody can use the model correctly. All numbers are read from your `metrics.json`, never typed by hand.
It also says plainly what the model is *not* good at (abstaining, unseen tools): an honest card is more convincing than a perfect-looking one.


In [ ]:
from src.prompts import SYSTEM_PROMPT

GITHUB_URL = "https://github.com/SwapnilGEU/SLM-Finetuning"   # <-- your repo

m = json.loads((RESULTS_DIR / "metrics.json").read_text())
s, vs, gb, bd = m["summary"], m["vs_base"], m["general_behaviour"], m["breakdowns"]
t = m["training"]
row = lambda k: f"| {k} | {s[k]['value']:.1f}% | {s[k]['ci_low']:.1f}–{s[k]['ci_high']:.1f} |"
pat = lambda k: f"| {k} | {bd['call_pattern'][k]['n']:,} | {bd['call_pattern'][k]['exact_match']:.1f}% |"

card = f"""---
base_model: Qwen/Qwen2.5-0.5B-Instruct
library_name: transformers
license: apache-2.0
pipeline_tag: text-generation
language: [en]
tags: [function-calling, tool-use, qlora, unsloth, small-language-model]
datasets: [Salesforce/xlam-function-calling-60k]
---

# {HF_REPO_ID.split("/")[-1]}

**Qwen2.5-0.5B-Instruct** fine-tuned with **QLoRA (Unsloth)** for **function calling**.
Give it a list of tools and a request; it returns the tool call(s) to make as a **JSON array**.
Small enough to run on a laptop CPU or a 6 GB GPU.

Code, evaluation and serving stack (vLLM + LangGraph gateway with guardrails + Langfuse): {GITHUB_URL}

## Results
Frozen test set of **{m["n"]:,}** xLAM questions, greedy decoding, scored by the same code for base and fine-tuned model.

| Metric | Fine-tuned | 95% CI |
|---|---|---|
{row("exact_match")}
{row("function_match")}
{row("argument_f1")}
{row("schema_valid")}

Exact match: base **{vs["a"]}%** → fine-tuned **{vs["b"]}%** ({vs["fixed"]} questions fixed, {vs["broke"]} broken, McNemar p < 0.001).

| Call pattern | n | Exact match |
|---|---|---|
{pat("single")}
{pat("parallel_same_tool")}
{pat("multi_distinct_tools")}
{pat("mixed")}

## Prompt format (important)
The model was trained on **this exact format**. It is *not* Qwen's native `<tool_call>` format, and it does not read OpenAI-style `tools=` parameters.

**System:**
```
{SYSTEM_PROMPT.strip()}
```
**User:**
```
Available tools:

<tools as a JSON list: name, description, parameters>

User request:

<the request>

Return the required tool call(s) as JSON.
```
**Assistant:** `[{{"name": "get_weather", "arguments": {{"city": "Pune"}}}}]`

## Usage
### transformers
```python
import json
from transformers import AutoModelForCausalLM, AutoTokenizer

repo = "{HF_REPO_ID}"
tok = AutoTokenizer.from_pretrained(repo)
model = AutoModelForCausalLM.from_pretrained(repo, dtype="auto", device_map="auto")

SYSTEM_PROMPT = \"\"\"<copy the system prompt above>\"\"\"
tools = [{{"name": "get_weather", "description": "Current weather for a city.",
          "parameters": {{"city": {{"type": "str", "description": "City name"}}}}}}]
user = (f"Available tools:\\n\\n{{json.dumps(tools, indent=2)}}\\n\\n"
        f"User request:\\n\\nWhat's the weather in Pune?\\n\\n"
        f"Return the required tool call(s) as JSON.")

messages = [{{"role": "system", "content": SYSTEM_PROMPT}}, {{"role": "user", "content": user}}]
inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                 return_tensors="pt", return_dict=True).to(model.device)
out = model.generate(**inputs, max_new_tokens=256)          # greedy is the default
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
```

### vLLM (OpenAI-compatible server)
```bash
vllm serve {HF_REPO_ID} --max-model-len 4096
```
Send the same two messages to `/v1/chat/completions` with **`temperature=0`**, then `json.loads` the reply.

### Decoding
Use **greedy decoding**. `generation_config.json` already sets `do_sample=False`; with OpenAI-compatible
servers pass `temperature=0`. Sampling and repetition penalties hurt JSON accuracy.

## How it behaves
- **With the system prompt above:** always answers with a JSON array of tool calls. It's a tool-calling specialist, not a chatbot.
- **Without it** (plain chat): answers in normal text. In our checks {gb["chat_as_tool_call_pct"]:.0f}% of plain chat messages turned into tool calls.
- **It does not execute anything.** It only writes the call; your code runs the tool. Validate the JSON against your tool
  schemas before running it, and ask the user before running tools with side effects.

## Limitations
- **Doesn't abstain:** when no tool fits, the right answer is `[]`, but it returned `[]` in {gb["irrelevance_returns_empty_pct"]:.0f}% of such cases.
  It picks the closest tool and may invent argument values. Guard against this (e.g. check arguments appear in the request).
- **Unseen tools:** almost every tool in the test set also appears in training (only a handful are new), so the score measures
  new *requests* for familiar tools. Accuracy on completely new tool schemas may be lower.
- Most errors are **wrong argument values** ({m["failure_mix"]["wrong_value"]:.0f}% of questions), e.g. formats or units, not wrong tools.
- Single turn only: it picks the call; it doesn't read tool results or write a final natural-language answer.
- English only; trained on xLAM's API-style tools.

## Training
QLoRA (4-bit base) with Unsloth, r={t["lora_r"]}, alpha={t["lora_alpha"]}, all linear layers, {t["epochs"]} epoch on ~48.7k xLAM examples,
effective batch {t["effective_batch"]}, lr {t["learning_rate"]}, max length {t["max_seq_length"]}, loss on the answer only.
LoRA merged into the bf16 base model.
"""
(MERGED_DIR / "README.md").write_text(card, encoding="utf-8")
print(card[:2500])


## 6. Log in
Paste your **write** token when the box appears. It's saved on your PC, so you only do this once.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## 7. Upload
`create_repo` makes the repo (`exist_ok=True`: re-running just updates it), and `upload_folder` sends the whole `merged/` folder, including the new `README.md` and `generation_config.json`. ~1 GB takes a few minutes.

Flip `PRIVATE = False` in step 1 when you're ready to make it public, or change it later on the website (Settings → visibility).


In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(HF_REPO_ID, private=PRIVATE, exist_ok=True)
api.upload_folder(folder_path=str(MERGED_DIR), repo_id=HF_REPO_ID,
                  commit_message=f"Upload {RUN_NAME} (merged QLoRA)")
print(f"✅ Done: https://huggingface.co/{HF_REPO_ID}")

## 8. Check: load it back from the Hub
This downloads the model the way any user would and runs the same test as step 4, **without** passing `do_sample`, so it also proves the greedy default from step 3 made it to the Hub.


In [ ]:
tok_hub = AutoTokenizer.from_pretrained(HF_REPO_ID)
model_hub = AutoModelForCausalLM.from_pretrained(HF_REPO_ID, dtype="auto", device_map="auto")
print("Hub generation config: do_sample =", model_hub.generation_config.do_sample)

inputs = tok_hub.apply_chat_template(build_messages(record), add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(model_hub.device)
out = getattr(model_hub, "generate")(**inputs, max_new_tokens=128)   # no do_sample: uses the repo default
print(tok_hub.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
